In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 读取csv文件
df = pd.read_csv('../results/run_1/summary_avg.csv')

# 设置美观的图形风格
sns.set(style="whitegrid")

# 熱力圖需要轉置數據使得每種算法成为一列
df_transposed = df.set_index('Scheduler').transpose()

# 绘制热图并保存
plt.figure(figsize=(10, 8))
sns.heatmap(df_transposed, annot=True, fmt=".2f", cmap='viridis')
plt.title('Comparison of Scheduling Algorithms')
plt.savefig('comparison_of_scheduling_algorithms_heatmap.png') # 保存热图
plt.show()

# 对于每个metric，绘制柱状图并保存
metrics = df.columns[1:] # 假设第一列是算法名称，跳过
for metric in metrics:
    plt.figure(figsize=(10, 6))
    sns.barplot(x='Scheduler', y=metric, data=df)
    plt.title(f'Comparison of {metric}')
    plt.xticks(rotation=45)
    plt.tight_layout() # 自动调整子图参数,使之填充整个图像区域
    
    # 保存当前metric的柱状图
    # plt.savefig(f'comparison_of_{metric.lower()}.png')
    
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 读取csv文件
df = pd.read_csv('../results/run_1/summary_avg.csv')

# 设置图像大小并绘制AvgLoadBalance折线图
plt.figure(figsize=(10, 6))
plt.plot(df['Scheduler'], df['AvgLoadBalance'], label='Average Load Balance', marker='o')
plt.title('Average Load Balance per Scheduler')
plt.xlabel('Scheduler')
plt.ylabel('Average Load Balance')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
# 保存AvgLoadBalance图像到本地
plt.savefig('avgloadbalance_per_scheduler.png')
plt.show()

# 设置图像大小并绘制AvgCost折线图
plt.figure(figsize=(10, 6))
plt.plot(df['Scheduler'], df['AvgCost'], label='Average Cost', marker='x')
plt.title('Average Cost per Scheduler')
plt.xlabel('Scheduler')
plt.ylabel('Average Cost')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
# 保存AvgCost图像到本地
plt.savefig('avgcost_per_scheduler.png')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 参数设置（与 Java 代码一致）
NUM_CLOUDLETS = 1000  # 模拟 1000 个云任务
MEAN_EXEC_TIME = 30000.0
VARIANCE_EXEC_TIME = 1.5

MEAN_FILE_SIZE = 100.0
VARIANCE_FILE_SIZE = 20.0

MEAN_OUTPUT_SIZE = 100.0
VARIANCE_OUTPUT_SIZE = 20.0

# 设置随机种子以便复现（可选）
np.random.seed(42)

# 1. 生成任务长度（对数正态分布）
# 注意：lognormal(mean, sigma) 中的 mean 是 μ（即 ln(MEAN_EXEC_TIME)），sigma 是 σ
mu_length = np.log(MEAN_EXEC_TIME)
sigma_length = VARIANCE_EXEC_TIME
lengths = np.random.lognormal(mean=mu_length, sigma=sigma_length, size=NUM_CLOUDLETS).astype(int)

# 2. 生成文件大小（截断正态分布，下限为 10）
file_sizes = np.random.normal(loc=MEAN_FILE_SIZE, scale=VARIANCE_FILE_SIZE, size=NUM_CLOUDLETS)
file_sizes = np.clip(file_sizes, a_min=10, a_max=None).astype(int)

# 3. 生成输出大小（同上）
output_sizes = np.random.normal(loc=MEAN_OUTPUT_SIZE, scale=VARIANCE_OUTPUT_SIZE, size=NUM_CLOUDLETS)
output_sizes = np.clip(output_sizes, a_min=10, a_max=None).astype(int)

# 创建子图
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# 任务长度分布（对数正态）
axs[0].hist(lengths, bins=50, color='skyblue', edgecolor='black')
axs[0].set_title('Task Length Distribution (Log-normal)', fontsize=14)
axs[0].set_xlabel('Length (MI)')
axs[0].set_ylabel('Frequency')
axs[0].grid(True, linestyle='--', alpha=0.6)

# 文件大小分布（截断正态）
axs[1].hist(file_sizes, bins=30, color='lightgreen', edgecolor='black')
axs[1].set_title('Input File Size Distribution (Truncated Normal)', fontsize=14)
axs[1].set_xlabel('File Size (units)')
axs[1].set_ylabel('Frequency')
axs[1].grid(True, linestyle='--', alpha=0.6)

# 输出大小分布（截断正态）
axs[2].hist(output_sizes, bins=30, color='salmon', edgecolor='black')
axs[2].set_title('Output File Size Distribution (Truncated Normal)', fontsize=14)
axs[2].set_xlabel('Output Size (units)')
axs[2].set_ylabel('Frequency')
axs[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('cloudlet_sci_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def plot_schedulers(directory):
    # 任务规模列表
    task_counts = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000]
    file_names = [f"{t}任务.csv" for t in task_counts]
    file_paths = [os.path.join(directory, fname) for fname in file_names]

    # 要绘制的指标
    metrics = ['AvgMakespan', 'AvgTotalTime', 'AvgLoadBalance', 'AvgCost']
    algorithms_data = {metric: {} for metric in metrics}

    # 要忽略的算法
    ignore_algorithms = {'HSO', 'HHO'}

    # 读取每个文件并提取数据
    for i, file in enumerate(file_paths):
        if not os.path.exists(file):
            print(f"⚠️ 警告：{file} 不存在，跳过。")
            continue

        df = pd.read_csv(file)

        # 过滤掉不需要的算法（大小写敏感，请确保 Scheduler 列中的名称一致）
        df = df[~df['Scheduler'].isin(ignore_algorithms)].reset_index(drop=True)

        n_tasks = task_counts[i]

        for _, row in df.iterrows():
            algo = row['Scheduler']
            for metric in metrics:
                if algo not in algorithms_data[metric]:
                    algorithms_data[metric][algo] = []
                algorithms_data[metric][algo].append(row[metric])

    # 开始绘图
    plt.rcParams.update({'font.size': 10})
    colors = plt.cm.tab10.colors  # 自动分配颜色

    for metric in metrics:
        plt.figure(figsize=(10, 6))
        algos = list(algorithms_data[metric].keys())

        for idx, algo in enumerate(algos):
            values = algorithms_data[metric][algo]
            if len(values) != len(task_counts):
                print(f"⚠️ 警告：算法 {algo} 在指标 {metric} 中数据不完整（应有 {len(task_counts)} 个点，实际 {len(values)}），跳过。")
                continue
            plt.plot(task_counts, values, marker='o', label=algo, color=colors[idx % len(colors)])

        plt.xlabel("Number of Tasks")
        plt.ylabel(metric)
        plt.title(f"Comparison of Scheduling Algorithms – {metric}")
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()

        # 保存图像到指定目录
        output_path = os.path.join(directory, f"{metric}_comparison.png")
        plt.savefig(output_path, dpi=300)
        print(f"✅ 已保存图像：{output_path}")

        plt.show()

# ==============================
# 使用示例：
# 请将下面路径替换为你自己的 CSV 文件所在文件夹路径
# ==============================
if __name__ == "__main__":
    plot_schedulers(r"C:\Users\LYL\Desktop\中期报告\结果\单目标\总结果")  # ← 修改这里！